In [0]:
import requests
import json
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
%sql
CREATE catalog if not EXISTS workspace;
CREATE SCHEMA if not EXISTS workspace.default;
CREATE volume if not EXISTS workspace.default.cricket_api_project;



In [0]:
base_path='/Volumes/workspace/default/cricket_api_project'


In [0]:
API_KEY='5651d3ff-9807-49f2-918e-4bfc982c1a3d'
api_url=f'https://api.cricapi.com/v1/currentMatches?apikey={API_KEY}&offset=0'

response= requests.get(api_url)
response.raise_for_status()

api_data=response.json()

print(json.dumps(api_data,indent=2)[:2000])


In [0]:
raw_file_path=f'{base_path}/current_matches_raw.json'

with open (raw_file_path,'w') as file:
  json.dump(api_data,file)

print("RAW API data is save at :", raw_file_path)

In [0]:
bronze_data=[{
    'source_api':api_url,
    'raw_json':json.dumps(api_data),
    'ingestion_time':None
}]

bronze_schema=StructType([
    StructField('source_api', StringType(),True),
    StructField('raw_json', StringType(),True),
    StructField('ingestion_time', TimestampType(),True)
])

bronze_df= spark.createDataFrame(bronze_data,bronze_schema).withColumn('ingestion_time',current_timestamp())

display(bronze_df)


bronze_df.write.mode('overwrite').format('delta').saveAsTable('workspace.default.cricket_bronze_current_matches')